# Isolate Corpus Files Referenced by `benchmarks_50`

Identifies and copies the exact corpus documents referenced by the 200 queries
in `data/LegalBenchRAG/benchmarks_50/` into a new directory
`data/LegalBenchRAG/corpus_50/`.

This subset is useful for fast ingestion runs that only need to cover the
50-query evaluation set, without indexing the full corpus (which contains
many documents never referenced by any benchmark query).

**Output**: `data/LegalBenchRAG/corpus_50/<benchmark>/<file>.txt`  

In [1]:
import json
import shutil
import pathlib
import urllib.parse
import pandas as pd

In [4]:
BENCHMARKS_DIR = pathlib.Path("data/LegalBenchRAG/benchmarks_50")
CORPUS_DIR     = pathlib.Path("data/LegalBenchRAG/corpus")
OUTPUT_DIR     = pathlib.Path("data/LegalBenchRAG/corpus_50")

DATASETS = ["contractnli", "cuad", "maud", "privacy_qa"]

In [5]:
def collect_referenced_files(dataset: str) -> set[str]:
    """Return the set of file_path values (relative, as stored in JSON) for a dataset."""
    path = BENCHMARKS_DIR / f"{dataset}.json"
    tests = json.loads(path.read_text())["tests"]
    files: set[str] = set()
    for test in tests:
        for snippet in test.get("snippets", []):
            files.add(snippet["file_path"])
    return files


referenced: dict[str, set[str]] = {}
for ds in DATASETS:
    referenced[ds] = collect_referenced_files(ds)
    print(f"{ds:>15s}: {len(referenced[ds])} unique files referenced")

total_unique = sum(len(v) for v in referenced.values())
print(f"\nTotal unique corpus files across all benchmarks: {total_unique}")

    contractnli: 36 unique files referenced
           cuad: 47 unique files referenced
           maud: 43 unique files referenced
     privacy_qa: 7 unique files referenced

Total unique corpus files across all benchmarks: 133


In [6]:
def resolve_corpus_path(file_path: str) -> pathlib.Path | None:
    """
    The JSON stores paths like ``contractnli/Some%20File.txt``.
    Try the literal path first, then URL-decode as a fallback.
    Returns the absolute Path if found, else None.
    """
    candidate = CORPUS_DIR / file_path
    if candidate.exists():
        return candidate
    decoded = CORPUS_DIR / urllib.parse.unquote(file_path)
    if decoded.exists():
        return decoded
    return None


stats = []
resolved: dict[str, dict[str, pathlib.Path]] = {}  # dataset -> {file_path -> src_path}

for ds in DATASETS:
    corpus_total = len(list((CORPUS_DIR / ds).glob("*.txt")))
    found, missing = {}, []
    for fp in sorted(referenced[ds]):
        src = resolve_corpus_path(fp)
        if src:
            found[fp] = src
        else:
            missing.append(fp)
    resolved[ds] = found
    stats.append({
        "dataset":       ds,
        "corpus_total":  corpus_total,
        "referenced":    len(referenced[ds]),
        "resolved":      len(found),
        "missing":       len(missing),
        "coverage_%":    round(len(found) / corpus_total * 100, 1),
    })
    if missing:
        print(f"[{ds}] {len(missing)} file(s) NOT found in corpus:")
        for m in missing:
            print(f"  {m}")

df_stats = pd.DataFrame(stats)
df_stats

,dataset,corpus_total,referenced,resolved,missing,coverage_%
0,contractnli,95,36,36,0,37.9
1,cuad,462,47,47,0,10.2
2,maud,150,43,43,0,28.7
3,privacy_qa,7,7,7,0,100.0


In [7]:
copied = 0
skipped = 0

for ds, file_map in resolved.items():
    out_sub = OUTPUT_DIR / ds
    out_sub.mkdir(parents=True, exist_ok=True)

    for fp, src in file_map.items():
        # Preserve the original filename (URL-decoded)
        dest = out_sub / src.name
        if dest.exists() and dest.stat().st_size == src.stat().st_size:
            skipped += 1
            continue
        shutil.copy2(src, dest)
        copied += 1

print(f"Copied  : {copied} file(s)")
print(f"Skipped : {skipped} file(s) (already present, same size)")
print(f"Output  : {OUTPUT_DIR.resolve()}")

Copied  : 133 file(s)
Skipped : 0 file(s) (already present, same size)
Output  : /home/twa174/LegalRAG/data/LegalBenchRAG/corpus_50


In [9]:
rows = []
for ds in DATASETS:
    bench_data = json.loads((BENCHMARKS_DIR / f"{ds}.json").read_text())["tests"]
    # Build file -> query count map
    query_count: dict[str, int] = {}
    for test in bench_data:
        touched = {s["file_path"] for s in test.get("snippets", [])}
        for fp in touched:
            query_count[fp] = query_count.get(fp, 0) + 1

    for fp, src in resolved[ds].items():
        rows.append({
            "dataset":      ds,
            "file":         src.name,
            "queries":      query_count.get(fp, 0),
            "size_KB":      round(src.stat().st_size / 1024, 1),
            "dest":         str(OUTPUT_DIR / ds / src.name),
        })

df_detail = pd.DataFrame(rows).sort_values(["dataset", "queries"], ascending=[True, False])
df_detail.reset_index(drop=True, inplace=True)
print(f"{len(df_detail)} files total")
df_detail

133 files total


,dataset,file,queries,size_KB,dest
0,contractnli,JBF_NDA_rev-2017033-1.txt,4,10.5,data/LegalBenchRAG/corpus_50/contractnli/JBF_N...
1,contractnli,DoiT-ICN-NonDisclosure-Agreement.txt,3,3.9,data/LegalBenchRAG/corpus_50/contractnli/DoiT-...
2,contractnli,FNHA-2019RFP-02-NDA-form.txt,3,5.1,data/LegalBenchRAG/corpus_50/contractnli/FNHA-...
3,contractnli,01_Bosch-Automotive-Service-Solutions-Mutual-N...,2,15.0,data/LegalBenchRAG/corpus_50/contractnli/01_Bo...
4,contractnli,IGC-Non-Disclosure-Agreement-LSE-Sample.txt,2,16.7,data/LegalBenchRAG/corpus_50/contractnli/IGC-N...
...,...,...,...,...,...
128,privacy_qa,Wordscapes.txt,9,27.4,data/LegalBenchRAG/corpus_50/privacy_qa/Wordsc...
129,privacy_qa,Groupon.txt,7,26.8,data/LegalBenchRAG/corpus_50/privacy_qa/Groupo...
130,privacy_qa,"TickTick: To Do List with Reminder, Day Planne...",7,3.1,data/LegalBenchRAG/corpus_50/privacy_qa/TickTi...
131,privacy_qa,Fiverr.txt,4,24.8,data/LegalBenchRAG/corpus_50/privacy_qa/Fiverr...
